In [26]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from category_encoders import OrdinalEncoder
from matplotlib.pylab import rand

честно не понял какой код надо было -дописать- с лекции (пытался найти, не нашел)

In [27]:
class MyGBRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.f0 = None

    def fit(self, X, y):
        self.f0 = np.mean(y)
        F_m = np.full_like(y, self.f0, dtype=np.float64)
        
        for _m in range(self.n_estimators):
            residuals = y - F_m
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            F_m += self.learning_rate * tree.predict(X)

    def predict(self, X):
        predictions = np.full((X.shape[0],), self.f0, dtype=np.float64)
        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)
        return predictions
    

In [28]:
df = pd.read_csv('ensembles-data-1.csv')

In [29]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['target']), df['target'], test_size=0.2, random_state=42)

model = MyGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)

pred = model.predict(X_test)

e = mean_squared_error(y_test, pred)
print(f'Mean Squared Error: {e}')
print(f'R^2 Score: {r2_score(y_test, pred)}')

Mean Squared Error: 0.02492480990710092
R^2 Score: 0.9660116228539533


In [30]:
#с добавленными фичами
#subsample, colsample_bytree, feature_importances_, категориальные признаки


class MyGBRegressorAdvanced:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, subsample = 1.0, colsample_bytree = 1.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.trees = []
        self.tree_features = []
        self.f0 = None
        self.encoder = None
        self.cat_cols = []
        self.n_features_ = None

    #обработка категориальных признаков
    def _preprocess(self, X, is_fit=False):
        X_copied = X.copy()
        if is_fit:
            if isinstance(X, np.ndarray):
                self.cat_cols = [i for i in range(X.shape[1]) if isinstance(X[0, i], str)]
            else:
                self.cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

            if len(self.cat_cols) > 0:
                self.encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
                if isinstance(X, np.ndarray):
                    X_copied[:, self.cat_cols] = self.encoder.fit_transform(X[:, self.cat_cols])
                else:
                    X_copied[self.cat_cols] = self.encoder.fit_transform(X[self.cat_cols])

        else:
            if self.encoder is not None and len(self.cat_cols) > 0:
                if isinstance(X, np.ndarray):
                    X_copied[:, self.cat_cols] = self.encoder.transform(X[:, self.cat_cols])
                else:
                    X_copied[self.cat_cols] = self.encoder.transform(X[self.cat_cols])

        return np.array(X_copied, dtype=np.float64)
    
    
    def fit(self, X, y):
        #категориальные признаки - кодирование
        X_processed = X.copy()

        self.f0 = np.mean(y)
        F_m = np.full_like(y, self.f0, dtype=np.float64)
        n_samples, n_features = X_processed.shape

        for _m in range(self.n_estimators):
            residuals = y - F_m
            
            #subsamples
            if self.subsample < 1.0:
                indices = np.random.choice(n_samples, size=int(n_samples * self.subsample), replace=False)
                X_b = X_processed[indices]
                res_b = residuals[indices]
            else:
                X_b = X_processed
                res_b = residuals

            #cosample_bytree
            if self.colsample_bytree < 1.0:
                feature_indices = np.random.choice(n_features, size=int(n_features * self.colsample_bytree), replace=False)
            else:
                feature_indices = np.arange(n_features)

            X_b_sub = X_b[:, feature_indices]

            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=42)
            tree.fit(X_b_sub, res_b)

            F_m += self.learning_rate * tree.predict(X_b_sub)

            self.trees.append(tree)
            self.tree_features.append(feature_indices)

    def predict(self, X):
        X_processed = X.copy()
        predictions = np.full((X_processed.shape[0],), self.f0, dtype=np.float64)
        for tree, feature_indices in zip(self.trees, self.tree_features):
            predictions += self.learning_rate * tree.predict(X_processed[:, feature_indices])
        return predictions
    
    #feature_importances_
    @property
    def feature_importances_(self):
        if not self.trees:
            raise ValueError("The model has not been fitted yet.")
        
        importances = np.zeros(self.n_features_)

        for tree, feature_indices in zip(self.trees, self.tree_features):
            tree_importances = tree.feature_importances_
            for local_idx, global_idx in enumerate(feature_indices):
                importances[global_idx] += tree_importances[local_idx] * self.learning_rate

        #нормализация
        sum_importances = np.sum(importances)
        if (sum_importances > 0):
            importances /= sum_importances

        return importances
    

KeyError: "None of [Index([ 5801,  8299,  8245,  8458,  6311, 13377,  5024,  1196,  6398,  5261,\n       ...\n        7306, 10438,  9658, 10674,  9550,  2345,  2230,  5925, 11800,  8946],\n      dtype='int64', length=13209)] are in the [columns]"